<a href="https://colab.research.google.com/github/prakashgoud421/FragmentTransaction/blob/fragment/1234LangChain_Overview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# @title Default title text
!pip install -q llama-cpp-python langchain gradio sentence-transformers chromadb PyPDF2 langchain langchain-community pypdf transformers accelerate bitsandbytes sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 MB 9.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 

In [21]:
import os
import hashlib
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from llama_cpp import Llama
from langchain_community.llms import LlamaCpp


In [22]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
MODEL_PATH = "/content/drive/MyDrive/models/mistral-7b-instruct-v0.1.Q4_K_M.gguf"
MODEL_PATH2 = "/content/drive/MyDrive/models/llama-2-7b-chat.Q4_K_M.gguf"
UPLOAD_DIR = "/content/uploaded_pdfs"
DB_DIR = "/content/vector_db"

os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(DB_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
!ls -lh /content/drive/MyDrive/models/

total 4.1G
-rw------- 1 root root 4.1G Sep 27  2023 mistral-7b-instruct-v0.1.Q4_K_M.gguf


In [27]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if "mistral" in file and file.endswith(".gguf"):
            print(os.path.join(root, file))


/content/drive/MyDrive/models/mistral-7b-instruct-v0.1.Q4_K_M.gguf


In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path="/content/drive/MyDrive/models/mistral-7b-instruct-v0.1.Q4_K_M.gguf",
    n_ctx=2048,
    n_gpu_layers=40,  # you can adjust based on available GPU
    verbose=True
)


llama_model_loader: loaded meta data with 20 key-value pairs and 291 tensors from /content/drive/MyDrive/models/mistral-7b-instruct-v0.1.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.1
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv  

In [20]:

print("Model exists:", os.path.exists(MODEL_PATH2))

Model exists: False


In [5]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
qa_chain = None

<ipython-input-5-d9b4f503ee6e>:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or dat

In [6]:
model_path = "/content/drive/MyDrive/models/mistral-7b-instruct-v0.1.Q4_K_M.gguf"
llm = LlamaCpp(
    model_path= model_path,  # use a GPU-friendly one
    n_gpu_layers=-1,      # Enable GPU acceleration
    n_ctx=2048,
    temperature=0.7,
    max_tokens=512,
    verbose=True,
)

llama_model_loader: loaded meta data with 20 key-value pairs and 291 tensors from /content/drive/MyDrive/models/mistral-7b-instruct-v0.1.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.1
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv  

In [7]:
def get_md5(file_path):
    with open(file_path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

In [8]:
def load_or_create_vectorstore(pdf_path):
    file_hash = get_md5(pdf_path)
    persist_path = os.path.join(DB_DIR, file_hash)

    if os.path.exists(persist_path):
        db = Chroma(persist_directory=persist_path, embedding_function=embedding)
    else:
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        docs = splitter.split_documents(documents)
        db = Chroma.from_documents(docs, embedding, persist_directory=persist_path)
        db.persist()
    return db



In [9]:

def upload_file(file_path):
    global qa_chain

    if not os.path.exists(file_path):
        return "❌ File not found."

    try:
        db = load_or_create_vectorstore(file_path)
        retriever = db.as_retriever()

        # Limit token size for llama-cpp compatibility
        qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=retriever,
            return_source_documents=True,
            chain_type_kwargs={"verbose": True}  # Optional: for debugging
        )

        return f"✅ Uploaded and ready: {os.path.basename(file_path)}"

    except Exception as e:
        return f"❌ Error processing file: {str(e)}"




In [10]:
def ask_question(question):
    if not qa_chain:
        return "❗ Please upload a PDF first."
    output = qa_chain.invoke({"query": question})
    return output["result"]



In [12]:
!nvidia-smi


Wed Apr  9 11:55:11 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P0             30W /   70W |     234MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:

with gr.Blocks() as demo:
    gr.Markdown("## 📄 Chat with your PDF (Mistral + LangChain)")
    file_input = gr.File(label="Upload PDF", type="filepath", file_types=[".pdf"])
    upload_button = gr.Button("Upload PDF")
    upload_status = gr.Textbox(label="Upload Status")

    question_input = gr.Textbox(label="Ask a question")
    answer_output = gr.Textbox(label="Answer")

    upload_button.click(upload_file, inputs=file_input, outputs=upload_status)
    question_input.submit(ask_question, inputs=question_input, outputs=answer_output)

demo.launch(debug=True, share=True, inline=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://591bff9d2f0ae53bef.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/gradio/queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/blocks.py", line 2137, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/blocks.py", line 1663, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/anyio/to_thread.py", line 56, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Pahari school of Painting, as also for the elaborate style of painting embellished with gold, 
known as the Tanjore styles from South India. 
 
1.15   India's diplomatic and strategic engagement with the Western countries has gone through 
radical changes in recent times. India maintains solid bilateral relations with the United States, 
United Kingdom, Germany and Russia.  India's claim for a permanent seat in the  UN 
Security Council , as well as more responsibility in institutions such as the  International 
Monetary Fund (IMF)  and the  World Bank  is much dependent on good relation with the 
developed West.  
 
 
List of Tables in this Chapter 
Sr. No.  No. of Table  Name of Table 
1 1.1 India, G20 and The World 
2

llama_perf_context_print:        load time =  427015.22 ms
llama_perf_context_print: prompt eval time =  427014.83 ms /  1095 tokens (  389.97 ms per token,     2.56 tokens per second)
llama_perf_context_print:        eval time =   81135.12 ms /   119 runs   (  681.81 ms per token,     1.47 tokens per second)
llama_perf_context_print:       total time =  508282.43 ms /  1214 tokens



> Finished chain.

> Finished chain.
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://591bff9d2f0ae53bef.gradio.live
